In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from pyspark.sql import DataFrame
from pyspark.sql.functions import col, to_date
from etl.transformations.common import (
    create_spark,
    epoch_to_timestamp,
    read_clickhouse,
    read_incremental_parquet,
    write_clickhouse,
)
from etl.transformations.facts.common import (
    add_date_key,
    add_farm_key,
    add_time_key,
)
from etl.transformations.state import (
    get_watermark,
    set_watermark,
)

WATERMARK_PATH = "s3a://staging/_checkpoints/spark/fact_sensor_readings/watermark.json"

SOURCE_PATH = "s3a://staging/raw/kafka/sensor_readings/"

In [2]:
def transform_source(
    readings_df: DataFrame,
) -> DataFrame:
    """
    Normalize raw sensor readings for warehouse loading.
    """

    return readings_df.select(
        "farm_sensor_id",
        "farm_id",
        "sensor_type_id",
        "value",
        epoch_to_timestamp(
            readings_df.timestamp,
        ).alias("reading_ts"),
    ).withColumn(
        "reading_date",
        to_date("reading_ts"),
    )


def add_sensor_keys(
    readings_df: DataFrame,
    dim_sensor_df: DataFrame,
) -> DataFrame:
    """
    Resolve sensor surrogate keys from dim_sensor.

    Uses the sensor version active when the reading occurred.
    """

    return readings_df.join(
        dim_sensor_df,
        (
            (readings_df.farm_sensor_id == dim_sensor_df.sensor_id)
            & (readings_df.reading_ts >= dim_sensor_df.valid_from)
            & (readings_df.reading_ts < dim_sensor_df.valid_to)
        ),
        "left",
    ).select(
        readings_df["*"],
        dim_sensor_df.sensor_key,
        dim_sensor_df.sensor_type_key,
    )


def add_anomaly_flag(
    readings_df: DataFrame,
    dim_sensor_type_df: DataFrame,
) -> DataFrame:
    """
    Mark readings outside optimal sensor ranges.
    """

    return (
        readings_df.join(
            dim_sensor_type_df,
            (
                (readings_df.sensor_type_key == dim_sensor_type_df.sensor_type_key)
                & (readings_df.reading_ts >= dim_sensor_type_df.valid_from)
                & (readings_df.reading_ts < dim_sensor_type_df.valid_to)
            ),
            "left",
        )
        .withColumn(
            "is_anomaly",
            (
                (readings_df.value < dim_sensor_type_df.optimal_min)
                | (readings_df.value > dim_sensor_type_df.optimal_max)
            ).cast("integer"),
        )
        .select(
            readings_df["*"],
            "is_anomaly",
        )
    )


def transform_fact_sensor_readings(
    readings_df: DataFrame,
    dim_date_df: DataFrame,
    dim_farm_df: DataFrame,
    dim_sensor_df: DataFrame,
    dim_sensor_type_df: DataFrame,
) -> DataFrame:
    """
    Build fact_sensor_readings dataframe.
    """

    readings_df = transform_source(
        readings_df,
    )
    print("transform_source:", readings_df.count())

    readings_df = add_date_key(
        readings_df,
        dim_date_df,
        "reading_date",
    )
    print("add_date_key:", readings_df.count())

    readings_df = add_time_key(
        readings_df,
        "reading_ts",
    )
    print("add_time_key:", readings_df.count())

    readings_df = add_farm_key(
        readings_df,
        dim_farm_df,
        "reading_ts",
    )
    print("add_farm_key:", readings_df.count())

    readings_df = add_sensor_keys(
        readings_df,
        dim_sensor_df,
    )
    print("add_sensor_keys:", readings_df.count())

    readings_df = add_anomaly_flag(
        readings_df,
        dim_sensor_type_df,
    )
    print("add_anomaly_flag:", readings_df.count())

    readings_df.printSchema()

    return readings_df.select(
        "farm_key",
        "farm_id",
        "sensor_key",
        "sensor_type_key",
        "date_key",
        "time_key",
        "reading_ts",
        "reading_date",
        "value",
        "is_anomaly",
    )


def main():

    spark = create_spark(
        "fact_sensor_readings",
    )

    try:
        last_watermark = get_watermark(
            spark,
            WATERMARK_PATH,
        )

        readings_df, newest_watermark = read_incremental_parquet(
            spark,
            SOURCE_PATH,
            last_watermark,
            watermark_parser=lambda value: value.split("=")[1],
            directory_prefix="event_date=",
        )

        print(f"Newest watermark: {newest_watermark}")
        print(f"Raw rows: {readings_df.count()}")
        
        readings_df.groupBy().count().show()

        if readings_df is None:
            print("No new sensor reading batches.")
            return

        dim_date_df = read_clickhouse(
            spark,
            "dim_date",
        )

        dim_farm_df = read_clickhouse(
            spark,
            """
                (
                    SELECT *
                    FROM dim_farm FINAL
                ) AS dim_farm
            """,
        )

        dim_sensor_df = read_clickhouse(
            spark,
            """
                (
                    SELECT *
                    FROM dim_sensor FINAL
                ) AS dim_sensor
            """,
        )

        dim_sensor_type_df = read_clickhouse(
            spark,
            """
                (
                    SELECT *
                    FROM dim_sensor_type FINAL
                ) AS dim_sensor_type
            """,
        )

        fact_df = transform_fact_sensor_readings(
            readings_df,
            dim_date_df,
            dim_farm_df,
            dim_sensor_df,
            dim_sensor_type_df,
        )

        if fact_df.filter(col("farm_key").isNull()).limit(1).count():
            raise RuntimeError("One or more farm keys could not be resolved.")

        if fact_df.filter(col("sensor_key").isNull()).limit(1).count():
            raise RuntimeError("One or more sensor keys could not be resolved.")

        if fact_df.filter(col("date_key").isNull()).limit(1).count():
            raise RuntimeError("One or more date keys could not be resolved.")

        print("Rows to write:", fact_df.count())
        fact_df.printSchema()

        write_clickhouse(
            fact_df,
            "fact_sensor_readings",
        )

        set_watermark(
            spark,
            WATERMARK_PATH,
            newest_watermark,
        )

    finally:
        spark.stop()


if __name__ == "__main__":
    main()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/23 12:06:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/23 12:06:52 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


Newest watermark: 2026-07-23


Raw rows: 668079


+------+
| count|
+------+
|668079|
+------+



transform_source: 668079


...

add_farm_key: 668079


add_sensor_keys: 668079


add_anomaly_flag: 668079
root
 |-- farm_sensor_id: integer (nullable = true)
 |-- farm_id: integer (nullable = true)
 |-- sensor_type_id: integer (nullable = true)
 |-- value: double (nullable = true)
 |-- reading_ts: timestamp (nullable = true)
 |-- reading_date: date (nullable = true)
 |-- date_key: decimal(20,0) (nullable = true)
 |-- time_key: integer (nullable = true)
 |-- farm_key: decimal(20,0) (nullable = true)
 |-- sensor_key: decimal(20,0) (nullable = true)
 |-- sensor_type_key: decimal(20,0) (nullable = true)
 |-- is_anomaly: integer (nullable = true)



Rows to write: 668079
root
 |-- farm_key: decimal(20,0) (nullable = true)
 |-- farm_id: integer (nullable = true)
 |-- sensor_key: decimal(20,0) (nullable = true)
 |-- sensor_type_key: decimal(20,0) (nullable = true)
 |-- date_key: decimal(20,0) (nullable = true)
 |-- time_key: integer (nullable = true)
 |-- reading_ts: timestamp (nullable = true)
 |-- reading_date: date (nullable = true)
 |-- value: double (nullable = true)
 |-- is_anomaly: integer (nullable = true)



26/07/23 12:07:30 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
26/07/23 12:07:30 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
26/07/23 12:07:30 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
26/07/23 12:07:30 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
26/07/23 12:07:30 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
26/07/23 12:07:30 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
26/07/23 12:07:30 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
26/07/23 12:07:30 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
